# 🚚 EcoRoute AI — Objective 3: Route Optimization for Waste Collection
### Dynamic Priority Scoring, Haversine Distance Matrix, Nearest Neighbor & 2-Opt TSP
---
**Project:** EcoRoute AI (Smart IoT Waste Management System)  
**Objective 3:** Eliminate unnecessary truck dispatches and minimize fuel consumption by generating mathematically optimal collection routes:  
1. **Multi-Factor Priority Scoring:** Ranks bins using IoT fill level, hours since last pickup, and citizen overflow grievances.  
2. **Geospatial Distance Matrix:** Computes exact travel distances using the spherical **Haversine formula**.  
3. **Nearest Neighbor Heuristic:** Establishes a greedy baseline route visiting high-priority bins first.  
4. **2-Opt Local Search Optimization:** Eliminates crossing paths and loops to achieve up to 30% fuel & distance reduction.  

**Target Environment:** Google Colab (Python 3.10+ / Folium / Matplotlib / Pandas / NumPy)

## Step 1: Install Dependencies & Mount Google Drive

In [ ]:
!pip install --quiet folium pandas numpy matplotlib seaborn

from google.colab import drive
import os

drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/EcoRoute_AI/Objective3_RouteOpt'
os.makedirs(DRIVE_DIR, exist_ok=True)
print('📁 Google Drive Workspace ready at:', DRIVE_DIR)

## Step 2: Realistic Municipal Bin Fleet with Coordinates & Telemetry

In [ ]:
import pandas as pd
import numpy as np

DEPOT = {'id': 'DEPOT-MAIN', 'name': 'Central Waste Depot', 'lat': 12.9716, 'lon': 77.5946, 'fill_level': 0, 'urgency': 0}

bins_data = [
    {'id': 'DB-001', 'name': 'City Market Gate 1',       'lat': 12.9634, 'lon': 77.5772, 'fill_level': 88, 'hours_uncollected': 14, 'citizen_reports': 4},
    {'id': 'DB-002', 'name': 'MG Road Metro Station',    'lat': 12.9756, 'lon': 77.6066, 'fill_level': 94, 'hours_uncollected': 18, 'citizen_reports': 7},
    {'id': 'DB-003', 'name': 'Cubbon Park South Gate',   'lat': 12.9738, 'lon': 77.5913, 'fill_level': 42, 'hours_uncollected': 6,  'citizen_reports': 0},
    {'id': 'DB-004', 'name': 'Brigade Road Corner',      'lat': 12.9702, 'lon': 77.6083, 'fill_level': 85, 'hours_uncollected': 16, 'citizen_reports': 5},
    {'id': 'DB-005', 'name': 'Commercial Street Hub',    'lat': 12.9822, 'lon': 77.6081, 'fill_level': 79, 'hours_uncollected': 10, 'citizen_reports': 2},
    {'id': 'DB-006', 'name': 'Indiranagar 100ft Road',   'lat': 12.9784, 'lon': 77.6408, 'fill_level': 91, 'hours_uncollected': 20, 'citizen_reports': 6},
    {'id': 'DB-007', 'name': 'Koramangala 5th Block',    'lat': 12.9352, 'lon': 77.6245, 'fill_level': 96, 'hours_uncollected': 22, 'citizen_reports': 9},
    {'id': 'DB-008', 'name': 'Jayanagar 4th Block',      'lat': 12.9298, 'lon': 77.5833, 'fill_level': 35, 'hours_uncollected': 4,  'citizen_reports': 0},
    {'id': 'DB-009', 'name': 'Lalbagh West Gate',        'lat': 12.9467, 'lon': 77.5851, 'fill_level': 72, 'hours_uncollected': 11, 'citizen_reports': 1},
    {'id': 'DB-010', 'name': 'Majestic Bus Stand',       'lat': 12.9772, 'lon': 77.5713, 'fill_level': 98, 'hours_uncollected': 24, 'citizen_reports': 12},
    {'id': 'DB-011', 'name': 'Malleshwaram 8th Cross',   'lat': 13.0031, 'lon': 77.5705, 'fill_level': 55, 'hours_uncollected': 8,  'citizen_reports': 0},
    {'id': 'DB-012', 'name': 'Rajajinagar 1st Block',    'lat': 12.9904, 'lon': 77.5528, 'fill_level': 82, 'hours_uncollected': 15, 'citizen_reports': 3},
]

df_fleet = pd.DataFrame(bins_data)
df_fleet.to_csv(os.path.join(DRIVE_DIR, 'fleet_bins.csv'), index=False)
print(f'✅ Fleet initialized with {len(df_fleet)} dustbins.')
df_fleet.head()

## Step 3: Multi-Factor Priority Scoring & Dynamic Threshold Filtering

In [ ]:
df_fleet['priority_score'] = (
    0.50 * df_fleet['fill_level'] +
    0.30 * np.clip((df_fleet['hours_uncollected'] / 24.0) * 100.0, 0, 100) +
    0.20 * np.clip(df_fleet['citizen_reports'] * 10.0, 0, 100)
)

active_bins = df_fleet[(df_fleet['priority_score'] >= 60.0) | (df_fleet['fill_level'] >= 75.0)].copy()
active_bins = active_bins.sort_values('priority_score', ascending=False).reset_index(drop=True)

print(f'Total Fleet Size: {len(df_fleet)} bins')
print(f'Actionable Bins Requiring Pickup Today: {len(active_bins)} bins ({(len(active_bins)/len(df_fleet))*100:.1f}%)')
print(f'Skipped Clean Bins (Fuel Saved!): {len(df_fleet) - len(active_bins)} bins')
active_bins[['id', 'name', 'fill_level', 'citizen_reports', 'priority_score']]

## Step 4: Geospatial Haversine Distance Matrix

In [ ]:
import math

def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371.0
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = (math.sin(dlat / 2.0) ** 2 +
         math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) *
         math.sin(dlon / 2.0) ** 2)
    c = 2.0 * math.atan2(math.sqrt(a), math.sqrt(1.0 - a))
    return R * c

all_stops = [DEPOT] + active_bins.to_dict('records')
n_stops = len(all_stops)

dist_matrix = np.zeros((n_stops, n_stops))
for i in range(n_stops):
    for j in range(n_stops):
        if i == j:
            dist_matrix[i][j] = 0.0
        else:
            dist_matrix[i][j] = haversine_distance(
                all_stops[i]['lat'], all_stops[i]['lon'],
                all_stops[j]['lat'], all_stops[j]['lon']
            )

print(f'✅ Computed {n_stops}x{n_stops} Haversine Distance Matrix between Depot and stops.')

## Step 5: Nearest Neighbor Heuristic & 2-Opt Route Optimization

In [ ]:
def nearest_neighbor_route(dist_mat):
    n = len(dist_mat)
    visited = [False] * n
    route = [0]
    visited[0] = True
    
    current = 0
    for _ in range(n - 1):
        next_stop = None
        min_dist = float('inf')
        for j in range(n):
            if not visited[j] and dist_mat[current][j] < min_dist:
                min_dist = dist_mat[current][j]
                next_stop = j
        route.append(next_stop)
        visited[next_stop] = True
        current = next_stop
        
    route.append(0)
    return route

def calculate_route_distance(route, dist_mat):
    return sum(dist_mat[route[i]][route[i+1]] for i in range(len(route) - 1))

nn_route = nearest_neighbor_route(dist_matrix)
nn_distance = calculate_route_distance(nn_route, dist_matrix)
print(f'🚗 Nearest Neighbor Distance: {nn_distance:.2f} km')

def two_opt_optimize(route, dist_mat, max_iterations=500):
    best_route = route.copy()
    best_dist = calculate_route_distance(best_route, dist_mat)
    improved = True
    iteration = 0
    
    while improved and iteration < max_iterations:
        improved = False
        iteration += 1
        for i in range(1, len(best_route) - 2):
            for j in range(i + 1, len(best_route) - 1):
                new_route = best_route[:i] + best_route[i:j+1][::-1] + best_route[j+1:]
                new_dist = calculate_route_distance(new_route, dist_mat)
                if new_dist < best_dist - 1e-4:
                    best_route = new_route
                    best_dist = new_dist
                    improved = True
                    break
            if improved:
                break
                
    return best_route, best_dist

opt_route, opt_distance = two_opt_optimize(nn_route, dist_matrix)
savings_km = nn_distance - opt_distance
savings_pct = (savings_km / nn_distance) * 100.0

print(f'✨ 2-Opt Optimized Distance:    {opt_distance:.2f} km')
print(f'💰 Distance Reduction Achieved: {savings_km:.2f} km ({savings_pct:.1f}% improvement!)')

## Step 6: Visual Route Comparison (Matplotlib) & Interactive Folium Map

In [ ]:
import matplotlib.pyplot as plt
import folium
import urllib.request
import json

# --- 1. Fetch Real Road Network Geometry (Curves, Turns, Actual Streets) via OSRM ---
print("🛣️ Fetching real road driving path with street curves from Open Source Routing Machine (OSRM)...")
real_road_coords = []
actual_road_km = opt_distance

try:
    wp_str = ";".join(f"{all_stops[idx]['lon']},{all_stops[idx]['lat']}" for idx in opt_route)
    osrm_url = f"http://router.project-osrm.org/route/v1/driving/{wp_str}?overview=full&geometries=geojson"
    req = urllib.request.Request(osrm_url, headers={'User-Agent': 'EcoRouteAI-WasteMgt/1.0'})
    with urllib.request.urlopen(req, timeout=10) as response:
        osrm_data = json.loads(response.read().decode('utf-8'))
        
    if osrm_data.get('code') == 'Ok':
        # GeoJSON is [lon, lat], Folium requires [lat, lon]
        real_road_coords = [[coord[1], coord[0]] for coord in osrm_data['routes'][0]['geometry']['coordinates']]
        actual_road_km = osrm_data['routes'][0]['distance'] / 1000.0
        print(f"✅ Real Road Path Retrieved: {len(real_road_coords)} curved street points | Actual Road Driving Distance: {actual_road_km:.2f} km")
    else:
        real_road_coords = [[all_stops[idx]['lat'], all_stops[idx]['lon']] for idx in opt_route]
except Exception as e:
    print(f"Notice: OSRM live routing fallback to waypoint vectors ({e})")
    real_road_coords = [[all_stops[idx]['lat'], all_stops[idx]['lon']] for idx in opt_route]

# --- 2. Side-by-Side Matplotlib Route Plot ---
plt.figure(figsize=(16, 7))

# Subplot 1: Initial Nearest Neighbor (Straight line crossings)
plt.subplot(1, 2, 1)
nn_lats = [all_stops[idx]['lat'] for idx in nn_route]
nn_lons = [all_stops[idx]['lon'] for idx in nn_route]
plt.plot(nn_lons, nn_lats, color='#f59e0b', linestyle='-', linewidth=2.5, marker='o', markersize=6, alpha=0.8, label='Truck Path')
plt.scatter([all_stops[0]['lon']], [all_stops[0]['lat']], color='black', s=160, marker='s', zorder=5, label='Depot')
for seq, stop_idx in enumerate(nn_route[:-1]):
    stop = all_stops[stop_idx]
    if stop_idx != 0:
        plt.scatter(stop['lon'], stop['lat'], color='#ef4444', s=100, zorder=4)
        plt.text(stop['lon'] + 0.001, stop['lat'] + 0.001, f'#{seq}', fontsize=10, fontweight='bold')
plt.title(f'Initial Nearest Neighbor (With Route Crossings)\nTotal Travel: {nn_distance:.2f} km', fontsize=12, fontweight='bold')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.grid(True, alpha=0.3)
plt.legend()

# Subplot 2: 2-Opt Optimized Route (Real Curved Street Geometry)
plt.subplot(1, 2, 2)
if len(real_road_coords) > len(opt_route):
    road_lats = [c[0] for c in real_road_coords]
    road_lons = [c[1] for c in real_road_coords]
    plt.plot(road_lons, road_lats, color='#10b981', linestyle='-', linewidth=2.5, alpha=0.9, label='Real Curved Road Path')
else:
    opt_lats = [all_stops[idx]['lat'] for idx in opt_route]
    opt_lons = [all_stops[idx]['lon'] for idx in opt_route]
    plt.plot(opt_lons, opt_lats, color='#10b981', linestyle='-', linewidth=2.5, marker='o', markersize=6, alpha=0.8, label='Optimized Path')

plt.scatter([all_stops[0]['lon']], [all_stops[0]['lat']], color='black', s=160, marker='s', zorder=5, label='Depot')
for seq, stop_idx in enumerate(opt_route[:-1]):
    stop = all_stops[stop_idx]
    if stop_idx != 0:
        plt.scatter(stop['lon'], stop['lat'], color='#ef4444', s=100, zorder=4)
        plt.text(stop['lon'] + 0.001, stop['lat'] + 0.001, f'#{seq}', fontsize=10, fontweight='bold')
plt.title(f'2-Opt Optimized Real Road Route (Realistic Curves & Turns)\nTotal Road Distance: {actual_road_km:.2f} km', fontsize=12, fontweight='bold')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.grid(True, alpha=0.3)
plt.legend()

plt.tight_layout()
plot_path = os.path.join(DRIVE_DIR, 'route_optimization_comparison.png')
plt.savefig(plot_path, dpi=300)
plt.show()

# --- 3. Interactive Real Street Map (Curves, Turns, Zero API Keys, No Watermarks) ---
depot_loc = [DEPOT['lat'], DEPOT['lon']]
m = folium.Map(
    location=depot_loc,
    zoom_start=13,
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}',
    attr='Tiles &copy; Esri &mdash; World Street Map'
)

# Depot Marker (Garage)
folium.Marker(
    depot_loc,
    popup=f"<b>{DEPOT['name']}</b><br>Start & End Point (Garage)",
    tooltip='Central Waste Depot',
    icon=folium.Icon(color='black', icon='home')
).add_to(m)

# High-Priority Dustbin Stops
for seq, stop_idx in enumerate(opt_route[1:-1], start=1):
    stop = all_stops[stop_idx]
    folium.Marker(
        [stop['lat'], stop['lon']],
        popup=(f"<b>Stop #{seq}: {stop['name']}</b><br>"
               f"Bin ID: {stop['id']}<br>"
               f"Fill Level: {stop['fill_level']}%<br>"
               f"Priority Score: {stop['priority_score']:.1f}"),
        tooltip=f"#{seq}: {stop['name']} ({stop['fill_level']}%)",
        icon=folium.Icon(color='red', icon='trash')
    ).add_to(m)
    
    # High-visibility circle badge
    folium.CircleMarker(
        location=[stop['lat'], stop['lon']],
        radius=8,
        color='#991b1b',
        fill=True,
        fill_color='#ef4444',
        fill_opacity=0.85
    ).add_to(m)

# REAL CURVED ROAD PATH on the street map
folium.PolyLine(
    real_road_coords,
    color='#059669', # Emerald green
    weight=5.5,
    opacity=0.9,
    tooltip=f"Realistic Road Driving Path ({actual_road_km:.2f} km with actual street curves)"
).add_to(m)

html_map_path = os.path.join(DRIVE_DIR, 'optimized_collection_route.html')
m.save(html_map_path)
print(f'✅ Interactive Curved Road Route Map saved at: {html_map_path}')
m

## Step 7: Export Turn-by-Turn Manifest & Google Drive Artifacts

In [ ]:
manifest_rows = []
cumulative_dist = 0.0

for seq, (curr_idx, next_idx) in enumerate(zip(opt_route[:-1], opt_route[1:])):
    leg_dist = dist_matrix[curr_idx][next_idx]
    cumulative_dist += leg_dist
    stop = all_stops[next_idx]
    manifest_rows.append({
        'Stop_Number': seq + 1,
        'Location_Name': stop['name'],
        'Bin_ID': stop['id'],
        'Leg_Distance_km': round(leg_dist, 2),
        'Cumulative_km': round(cumulative_dist, 2),
        'Latitude': stop['lat'],
        'Longitude': stop['lon'],
        'Est_Fill_Level_Pct': stop.get('fill_level', 0)
    })

df_manifest = pd.DataFrame(manifest_rows)
manifest_csv_path = os.path.join(DRIVE_DIR, 'driver_dispatch_manifest.csv')
df_manifest.to_csv(manifest_csv_path, index=False)

print('='*70)
print('📋 DRIVER DISPATCH TURN-BY-TURN MANIFEST')
print('='*70)
print(df_manifest[['Stop_Number', 'Location_Name', 'Leg_Distance_km', 'Cumulative_km', 'Est_Fill_Level_Pct']].to_string(index=False))

print('\n' + '='*70)
print('🎉 OBJECTIVE 3 COMPLETED: ALL ARTIFACTS SAVED IN GOOGLE DRIVE!')
print('='*70)
print(f'Output files stored in: {DRIVE_DIR}')